In [ ]:
# mount the data from google drive
from google.colab import drive
from google.colab import files
drive.mount("/content/drive")

# navigate to the project folder
%cd drive/MyDrive/projects/CVPR

Mounted at /content/drive
/content/drive/MyDrive/projects/CVPR


In [ ]:
#best image plot
from PIL import Image

def plot_images_and_heatmaps(images, arrays, figsize=(12, 15), cmap='gist_heat', title_name=''):
    """
    Plot images in the top row and arrays as heatmaps in rows below.
    Arrays are arranged row by row across the grid.

    Parameters:
    -----------
    images : list
        List of 3 images to display in the top row
    arrays : list
        List of 12 numpy arrays to display as heatmaps (4 arrays × 3 rows)
    figsize : tuple, optional
        Figure size (width, height)
    cmap : str, optional
        Colormap to use for heatmaps

    Returns:
    --------
    fig : matplotlib figure
        The created figure object
    """
    if len(images) != 3:
        raise ValueError("Expected 3 images")
    if len(arrays) != 12:
        raise ValueError("Expected 12 arrays for heatmaps")

    # Create figure with 3 columns and 5 rows (1 row for images + 4 rows for heatmaps)
    fig, axes = plt.subplots(5, 3, figsize=figsize)

    # Plot images in the top row
    for i, img in enumerate(images):
        axes[0, i].imshow(img)
        axes[0, i].axis('off')

    # Plot heatmaps in the rows below, arranged row by row
    array_idx = 0
    for row in range(1, 5):  # Rows 1-4 for heatmaps
        for col in range(3):  # 3 columns
            if array_idx < len(arrays):
                axes[row, col].imshow(arrays[array_idx], cmap=cmap)
                axes[row, col].axis('off')
                array_idx += 1

    # Reduce white space
    fig.suptitle(title_name, fontsize=20)
    plt.subplots_adjust(wspace=0.01, hspace=0.01)

    return fig

def load_numpy_arrays(file_paths):
    """
    Takes a list of file paths to numpy arrays and returns a list of the loaded arrays.

    Parameters:
    file_paths (list of str): A list of file paths to numpy arrays.

    Returns:
    list of numpy.ndarray: A list containing the loaded numpy arrays.
    """

    loaded_arrays = [bethonio_gaussian(np.load(file)) for file in file_paths]
    return loaded_arrays

def load_images(image_paths):
    images = []
    for path in image_paths:
        try:
            img = Image.open(path)
            images.append(img)
        except Exception as e:
            print(f"Could not load image at {path}: {e}")
    return images

def list_files(path):
    """Lists all files in the specified directory."""

    files = []
    for entry in os.listdir(path):
        if os.path.isfile(os.path.join(path, entry)):
            files.append(entry[:-4])
    return files

img_dir = 'ms-coco/coco-images/'
mask_dir = 'ms-coco/object_arrays/'
attention_dir = 'ms-coco/fixations/'

best_im = [ '000000081074', '000000399164', '000000326210']

image_paths = [img_dir + item + '.jpg' for item in best_im]
values_paths = [attention_dir + item + '.npy' for item in best_im]

attention_list = load_numpy_arrays(values_paths)
images_list = load_images(image_paths)


pred_list = []
for image in best_im:

  masks = np.load(str(mask_dir + image + '.npy'))
  mask_ids = list(values[values['image']==int(image.lstrip('0'))]['id'])
  image_values = list(values[values['image']==int(image.lstrip('0'))]['pred'])

  pred_list.append(fill_masks_with_values(masks, mask_ids, image_values))

true_list = []
for image in best_im:

  masks = np.load(str(mask_dir + image + '.npy'))
  mask_ids = list(values[values['image']==int(image.lstrip('0'))]['id'])
  image_values = list(values[values['image']==int(image.lstrip('0'))]['sum'])

  true_list.append(fill_masks_with_values(masks, mask_ids, image_values))

res_list = []
for image in best_im:

  masks = np.load(str(mask_dir + image + '.npy'))
  mask_ids = list(values[values['image']==int(image.lstrip('0'))]['id'])
  image_values = list(values[values['image']==int(image.lstrip('0'))]['abs_diff'])

  res_list.append(fill_masks_with_values(masks, mask_ids, image_values))

arrays_list = pred_list + true_list + res_list + attention_list

fig = plot_images_and_heatmaps(images_list, arrays_list, figsize=(6, 7.5), cmap='gist_heat', title_name = 'Best Prediction')
#plt.savefig("figures/best_examples.png", dpi=1000)
#files.download("figures/best_examples.png")


In [ ]:
def plot_images_and_heatmaps(images, arrays, figsize=(12, 15), cmap='gist_heat', title_name=''):
    """
    Plot images in the top row and arrays as heatmaps in rows below.
    Arrays are arranged row by row across the grid.

    Parameters:
    -----------
    images : list
        List of 3 images to display in the top row
    arrays : list
        List of 12 numpy arrays to display as heatmaps (4 arrays × 3 rows)
    figsize : tuple, optional
        Figure size (width, height)
    cmap : str, optional
        Colormap to use for heatmaps

    Returns:
    --------
    fig : matplotlib figure
        The created figure object
    """
    if len(images) != 3:
        raise ValueError("Expected 3 images")
    if len(arrays) != 12:
        raise ValueError("Expected 12 arrays for heatmaps")

    # Create figure with 3 columns and 5 rows (1 row for images + 4 rows for heatmaps)
    fig, axes = plt.subplots(5, 3, figsize=figsize)

    # Plot images in the top row
    for i, img in enumerate(images):
        axes[0, i].imshow(img)
        axes[0, i].axis('off')

    # Plot heatmaps in the rows below, arranged row by row
    array_idx = 0
    for row in range(1, 5):  # Rows 1-4 for heatmaps
        for col in range(3):  # 3 columns
            if array_idx < len(arrays):
                axes[row, col].imshow(arrays[array_idx], cmap=cmap)
                axes[row, col].axis('off')
                array_idx += 1

    # Reduce white space
    fig.suptitle(title_name, fontsize=20)
    plt.subplots_adjust(wspace=0.01, hspace=0.01)

    return fig